# Experiment Summary Tables & Figures

Collect all saved metrics into thesis-ready tables and plots for Chapter 3.

Safe to run even if notebooks 06–08 are incomplete — missing files produce warnings only.  
Pretrained results include AST, MERT, and CLAP when available in `pretrained_audio_models_summary.csv`.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "configs").exists() and (PROJECT_ROOT.parent / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import get_project_root, load_configs, resolve_path
from src.reporting.collect_results import collect_experiment_summaries
from src.reporting.summary_tables import build_final_summary_tables
from src.reporting.result_plots import plot_final_result_figures

configs = load_configs(PROJECT_ROOT)
root = get_project_root(PROJECT_ROOT)

## 1. Collect available experiment summaries

In [2]:
combined, warnings = collect_experiment_summaries(configs)

print(f"Collected rows: {len(combined)}")
if warnings:
    print("\nWarnings:")
    for msg in warnings:
        print(f"  - {msg}")

pretrained_test = combined[
    (combined["experiment_group"] == "pretrained") & (combined["eval_split"] == "test")
]
if pretrained_test.empty:
    print("\nNo pretrained test results found yet (run notebook 08).")
else:
    print(f"\nPretrained test rows: {len(pretrained_test)}")
    display(
        pretrained_test.sort_values("macro_f1", ascending=False)[
            ["model_name", "accuracy", "macro_f1", "weighted_f1"]
        ].head(12)
    )

combined.head(10)

Collected rows: 74

Pretrained test rows: 12


,model_name,accuracy,macro_f1,weighted_f1
51,ast_audioset_logistic_regression,0.627306,0.541733,0.625412
57,ast_audioset_xgboost,0.708487,0.538109,0.656434
55,ast_audioset_gradient_boosting,0.675277,0.492451,0.623150
73,clap_htsat_unfused_xgboost,0.667897,0.479341,0.615731
67,clap_htsat_unfused_logistic_regression,0.586716,0.470764,0.578115
59,mert_v1_95m_logistic_regression,0.571956,0.446662,0.573177
71,clap_htsat_unfused_gradient_boosting,0.634686,0.444027,0.588929
69,clap_htsat_unfused_random_forest,0.664207,0.417533,0.577687
63,mert_v1_95m_gradient_boosting,0.616236,0.392660,0.545503
53,ast_audioset_random_forest,0.656827,0.379929,0.554888


,experiment_group,task_type,model_name,feature_type,target_type,eval_split,accuracy,macro_f1,weighted_f1,mae,rmse,r2,comments
0,static,static,logistic_regression,spectral_aggregated,emotion_quadrant,val,0.544444,0.384512,0.518591,NaN,NaN,NaN,Thresholds fit on train tracks only; splits by...
1,static,static,logistic_regression,spectral_aggregated,emotion_quadrant,test,0.564576,0.427884,0.545204,NaN,NaN,NaN,Thresholds fit on train tracks only; splits by...
2,static,static,knn,spectral_aggregated,emotion_quadrant,val,0.551852,0.423505,0.533473,NaN,NaN,NaN,Thresholds fit on train tracks only; splits by...
3,static,static,knn,spectral_aggregated,emotion_quadrant,test,0.564576,0.421810,0.539531,NaN,NaN,NaN,Thresholds fit on train tracks only; splits by...
4,static,static,svm,spectral_aggregated,emotion_quadrant,val,0.614815,0.366856,0.526352,NaN,NaN,NaN,Thresholds fit on train tracks only; splits by...
5,static,static,svm,spectral_aggregated,emotion_quadrant,test,0.612546,0.365864,0.523281,NaN,NaN,NaN,Thresholds fit on train tracks only; splits by...
6,static,static,random_forest,spectral_aggregated,emotion_quadrant,val,0.614815,0.377374,0.530224,NaN,NaN,NaN,Thresholds fit on train tracks only; splits by...
7,static,static,random_forest,spectral_aggregated,emotion_quadrant,test,0.608856,0.376117,0.525322,NaN,NaN,NaN,Thresholds fit on train tracks only; splits by...
8,static,static,gradient_boosting,spectral_aggregated,emotion_quadrant,val,0.611111,0.410433,0.549406,NaN,NaN,NaN,Thresholds fit on train tracks only; splits by...
9,static,static,gradient_boosting,spectral_aggregated,emotion_quadrant,test,0.571956,0.379244,0.517763,NaN,NaN,NaN,Thresholds fit on train tracks only; splits by...


## 2. Build and save final tables

In [3]:
outputs = build_final_summary_tables(configs, combined=combined)

for name, path in outputs["paths"].items():
    print(f"{name}: {path}")

outputs["best_model_per_task"]

final_experiment_summary: C:\Users\athen\Desktop\music-emotion-recognition\reports\tables\final_experiment_summary.csv
best_model_per_task: C:\Users\athen\Desktop\music-emotion-recognition\reports\tables\best_model_per_task.csv
static_vs_dynamic_comparison: C:\Users\athen\Desktop\music-emotion-recognition\reports\tables\static_vs_dynamic_comparison.csv
classical_vs_neural_comparison: C:\Users\athen\Desktop\music-emotion-recognition\reports\tables\classical_vs_neural_comparison.csv
feature_type_comparison: C:\Users\athen\Desktop\music-emotion-recognition\reports\tables\feature_type_comparison.csv
regression_results: C:\Users\athen\Desktop\music-emotion-recognition\reports\tables\regression_results.csv


,task_type,model_name,feature_type,accuracy,macro_f1,weighted_f1
0,dynamic_window_classification,logistic_regression,dynamic_window_spectral,0.600127,0.461823,0.572606
1,pretrained_embedding,ast_audioset_logistic_regression,pretrained_audio_embedding,0.627306,0.541733,0.625412
2,sequence,transformer,mfcc_sequence,0.594096,0.509829,0.586491
3,spectrogram,crnn,mel_spectrogram,0.638376,0.370703,0.540945
4,static,logistic_regression,spectral_aggregated,0.564576,0.427884,0.545204


In [4]:
outputs["static_vs_dynamic_comparison"]

,experiment_group,accuracy,macro_f1,weighted_f1
0,dynamic_window,0.603476,0.461823,0.572606
1,static,0.612546,0.427884,0.545204


In [5]:
outputs["classical_vs_neural_comparison"]

,model_family,accuracy,macro_f1,weighted_f1
0,classical,0.612546,0.461823,0.572606
1,neural,0.708487,0.541733,0.656434


In [6]:
outputs["feature_type_comparison"]

,feature_type,accuracy,macro_f1,weighted_f1
3,pretrained_audio_embedding,0.708487,0.541733,0.656434
2,mfcc_sequence,0.605166,0.509829,0.586491
0,dynamic_window_spectral,0.603476,0.461823,0.572606
4,spectral_aggregated,0.612546,0.427884,0.545204
1,mel_spectrogram,0.638376,0.370703,0.540945


## 3. Final comparison plots

In [7]:
figure_paths = plot_final_result_figures(configs, combined=combined)
for key, path in figure_paths.items():
    print(f"{key}: {path}")

C:\Users\athen\Desktop\music-emotion-recognition\src\reporting\result_plots.py:52: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha="right")
C:\Users\athen\Desktop\music-emotion-recognition\src\reporting\result_plots.py:52: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha="right")
C:\Users\athen\Desktop\music-emotion-recognition\src\reporting\result_plots.py:87: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha="right")


macro_f1: C:\Users\athen\Desktop\music-emotion-recognition\reports\figures\final_results\classification_macro_f1_comparison.png
accuracy: C:\Users\athen\Desktop\music-emotion-recognition\reports\figures\final_results\classification_accuracy_comparison.png
by_experiment_group: C:\Users\athen\Desktop\music-emotion-recognition\reports\figures\final_results\classification_by_experiment_group.png
regression_mae: C:\Users\athen\Desktop\music-emotion-recognition\reports\figures\final_results\regression_mae_comparison.png
regression_rmse: C:\Users\athen\Desktop\music-emotion-recognition\reports\figures\final_results\regression_rmse_comparison.png


C:\Users\athen\Desktop\music-emotion-recognition\src\reporting\result_plots.py:87: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha="right")
